# CS781 Phase 1 — Model 1 — Generate Codabench Submission

**Inputs to attach**: (1) the tokenizer dataset, (2) model 1's training notebook output (needs its `legal-gqa-final` folder with the final saved checkpoint), and (3) optionally, the Prepare Test Set notebook's output -- if attached, generation uses that fixed snapshot instead of downloading the test file fresh (which the dataset repo says is a dynamic, changing document).

Needs internet + HF login only as a fallback (if no snapshot is attached). No GPU-heavy work here compared to training -- no backward pass, and KV-caching makes each of the 128 decode steps cheap after the prompt prefill.

## 1. Hugging Face Login

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Hugging Face login successful.")

## 2. Check GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 3. Configuration

In [ ]:
import glob, json, math, time, zipfile

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"  # verified via list_repo_files() -- the earlier PDF-pasted path was corrupted

TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 256
TEMPERATURE = 0.7
TOP_P = 0.95
GENERATION_BATCH_SIZE = 32

OUTPUT_DIR = "/kaggle/working/cs781_phase1_model1_submission"
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 4. Load Test Data

Prefers an attached snapshot from the Prepare Test Set notebook (a fixed, known version of the test file) over downloading fresh -- the dataset repo notes the test file is dynamic and can change, so generating against whatever happened to be live at this exact moment isn't reproducible. Falls back to downloading directly if no snapshot is attached.

In [ ]:
def find_dirs(filename):
    return sorted({os.path.dirname(p) for p in glob.glob(f"/kaggle/input/**/{filename}", recursive=True)})

snapshot_dirs = find_dirs("test_snapshot.json")
if snapshot_dirs:
    test_file_path = os.path.join(snapshot_dirs[0], "test_snapshot.json")
    print("Using attached snapshot:", test_file_path)
    stats_path = os.path.join(snapshot_dirs[0], "test_stats.json")
    if os.path.exists(stats_path):
        print("Snapshot info:", json.load(open(stats_path)))
else:
    from huggingface_hub import hf_hub_download
    print("No snapshot attached -- downloading fresh (this will be whatever is live right now).")
    test_file_path = hf_hub_download(
        repo_id=HF_DATASET_REPO, filename=TEST_FILE, repo_type="dataset", token=HF_TOKEN
    )
    print("Downloaded:", test_file_path)

with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)
print("Number of test examples:", len(test_data))

## 5. Load Tokenizer

Same shared tokenizer as training -- searches `/kaggle/input` for it, so the exact mount path doesn't matter.

In [ ]:
from transformers import AutoTokenizer

def find_dirs(filename):
    return sorted({os.path.dirname(p) for p in glob.glob(f"/kaggle/input/**/{filename}", recursive=True)})

cands = [d for d in find_dirs("tokenizer.json") if "checkpoint-" not in d and "legal-gqa-final" not in d]
if not cands:
    raise FileNotFoundError("No tokenizer dataset found -- attach it via Add Input.")
TOKENIZER_DIR = ([d for d in cands if os.path.exists(os.path.join(d, "trained_on.json"))] or cands)[0]

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR)
tokenizer.padding_side = "left"  # required for correct batched generation
print("Tokenizer:", TOKENIZER_DIR, "| vocab size:", len(tokenizer))

## 6. Define The Model

Written to a real file and imported, same reason as the training notebook: `transformers` reads `__file__` off a model class's module, and a class defined live in a notebook cell has none.

In [ ]:
%%writefile /kaggle/working/legal_gqa_model.py
"""
Model 1 for CS781 Phase-1: GQA + RoPE + SwiGLU + RMSNorm decoder-only transformer.

Wrapped as a transformers PretrainedConfig/PreTrainedModel subclass so it is a
drop-in replacement for GPT2Config/GPT2LMHeadModel in the starter notebook:
same save_pretrained/from_pretrained, same Trainer + TrainingArguments +
resume_from_checkpoint behavior, same model.generate() call signature.
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as torch_checkpoint
from transformers import PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithPast


class LegalGQAConfig(PretrainedConfig):
    model_type = "legal_gqa"

    def __init__(
        self,
        vocab_size=50257,
        n_positions=1024,
        n_embd=1024,
        n_layer=8,
        n_head=16,
        n_kv_head=4,
        d_ff=2688,
        rope_theta=10000.0,
        dropout=0.1,
        initializer_range=0.02,
        rms_norm_eps=1e-5,
        bos_token_id=50256,
        eos_token_id=50256,
        pad_token_id=50256,
        activation_checkpointing=False,
        tie_word_embeddings=True,
        **kwargs,
    ):
        assert n_embd % n_head == 0, "n_embd must be divisible by n_head"
        assert n_head % n_kv_head == 0, "n_head must be divisible by n_kv_head"
        self.vocab_size = vocab_size
        self.n_positions = n_positions
        self.n_embd = n_embd
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.d_ff = d_ff
        self.rope_theta = rope_theta
        self.dropout = dropout
        self.initializer_range = initializer_range
        self.rms_norm_eps = rms_norm_eps
        self.use_cache = True
        # own implementation (HF's gradient_checkpointing flag is not supported here)
        self.activation_checkpointing = activation_checkpointing
        super().__init__(
            bos_token_id=bos_token_id,
            eos_token_id=eos_token_id,
            pad_token_id=pad_token_id,
            tie_word_embeddings=tie_word_embeddings,
            **kwargs,
        )


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(q, k, cos, sin):
    # q, k: (B, n_head, T, head_dim); cos/sin: (T, head_dim)
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    q_rot = (q * cos) + (rotate_half(q) * sin)
    k_rot = (k * cos) + (rotate_half(k) * sin)
    return q_rot, k_rot


class GroupedQueryAttention(nn.Module):
    def __init__(self, config: LegalGQAConfig):
        super().__init__()
        self.n_head = config.n_head
        self.n_kv_head = config.n_kv_head
        self.head_dim = config.n_embd // config.n_head
        self.n_rep = self.n_head // self.n_kv_head
        self.dropout_p = config.dropout

        self.q_proj = nn.Linear(config.n_embd, config.n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=False)
        self.o_proj = nn.Linear(config.n_head * self.head_dim, config.n_embd, bias=False)

    def forward(self, x, cos, sin, attn_bias, is_causal, past_key_value=None, use_cache=False):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        q, k = apply_rope(q, k, cos, sin)
        # RoPE tables are fp32, v comes out of a linear (fp16 under autocast): force one dtype
        q, k = q.to(v.dtype), k.to(v.dtype)

        if past_key_value is not None:
            past_k, past_v = past_key_value
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
        present = (k, v) if use_cache else None

        # GQA: broadcast each KV head across its group of query heads
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)

        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_bias,
            dropout_p=self.dropout_p if self.training else 0.0,
            is_causal=is_causal,
        )
        out = out.transpose(1, 2).contiguous().view(B, T, self.n_head * self.head_dim)
        return self.o_proj(out), present


class SwiGLU(nn.Module):
    def __init__(self, config: LegalGQAConfig):
        super().__init__()
        self.gate_proj = nn.Linear(config.n_embd, config.d_ff, bias=False)
        self.up_proj = nn.Linear(config.n_embd, config.d_ff, bias=False)
        self.down_proj = nn.Linear(config.d_ff, config.n_embd, bias=False)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class DecoderBlock(nn.Module):
    def __init__(self, config: LegalGQAConfig):
        super().__init__()
        self.ln_1 = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.attn = GroupedQueryAttention(config)
        self.ln_2 = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.ffn = SwiGLU(config)

    def forward(self, x, cos, sin, attn_bias, is_causal, past_key_value=None, use_cache=False):
        attn_out, present = self.attn(
            self.ln_1(x), cos, sin, attn_bias, is_causal, past_key_value, use_cache
        )
        x = x + attn_out
        x = x + self.ffn(self.ln_2(x))
        return x, present


class LegalGQAForCausalLM(PreTrainedModel):
    config_class = LegalGQAConfig
    # input embedding and output head share one weight; must be declared or
    # save_pretrained / Trainer checkpointing refuses to save
    _tied_weights_keys = {"lm_head.weight": "wte.weight"}

    def __init__(self, config: LegalGQAConfig):
        super().__init__(config)
        self.head_dim = config.n_embd // config.n_head

        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([DecoderBlock(config) for _ in range(config.n_layer)])
        self.ln_f = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight  # weight tying

        self.post_init()

    def _init_weights(self, module):
        std = self.config.initializer_range
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=std)
        # scaled init on residual-stream output projections (GPT-2 recipe)
        for name, p in module.named_parameters(recurse=False):
            if name in ("o_proj.weight", "down_proj.weight"):
                nn.init.normal_(
                    p, mean=0.0, std=std / math.sqrt(2 * self.config.n_layer)
                )

    def get_input_embeddings(self):
        return self.wte

    def get_output_embeddings(self):
        return self.lm_head

    def set_output_embeddings(self, new_embeddings):
        self.lm_head = new_embeddings

    def set_input_embeddings(self, value):
        self.wte = value

    def _rope_slice(self, past_len, T, device, dtype):
        # Computed on the fly (float32, then cast). Deliberately NOT stored as a buffer:
        # non-persistent buffers come back uninitialized after from_pretrained().
        inv_freq = 1.0 / (
            self.config.rope_theta
            ** (torch.arange(0, self.head_dim, 2, device=device).float() / self.head_dim)
        )
        pos = torch.arange(past_len, past_len + T, device=device).float()
        emb = torch.cat((torch.outer(pos, inv_freq),) * 2, dim=-1)
        return emb.cos().to(dtype), emb.sin().to(dtype)

    def _build_attn_bias(self, attention_mask, past_len, T, device, dtype):
        """Returns (attn_bias, is_causal). attn_bias is None when the fast
        SDPA is_causal fused-kernel path is usable (no padding to mask)."""
        no_padding = attention_mask is None or bool(attention_mask.bool().all())
        if no_padding:
            return None, T > 1

        total_len = past_len + T
        pad = attention_mask[:, None, None, :].to(dtype)
        bias = (1.0 - pad) * torch.finfo(dtype).min

        if T > 1:
            q_pos = torch.arange(past_len, past_len + T, device=device).unsqueeze(1)
            k_pos = torch.arange(0, total_len, device=device).unsqueeze(0)
            causal = (k_pos > q_pos).to(dtype) * torch.finfo(dtype).min
            bias = bias + causal[None, None, :, :]
        return bias, False

    def forward(
        self,
        input_ids,
        attention_mask=None,
        past_key_values=None,
        labels=None,
        use_cache=None,
    ):
        # NOTE: no **kwargs here on purpose. If forward() accepts **kwargs, Trainer assumes the
        # model normalises its own loss, then multiplies it by n_gpu and skips the
        # gradient-accumulation division (loss reported 2x-64x too high).
        #
        # fp16 autocast is switched on HERE rather than relying on Trainer: under nn.DataParallel
        # on Kaggle the matmuls were measurably running in fp32 (no tensor cores, ~4x slower).
        # Only used for unpadded batches (all of training/eval); padded generation keeps the
        # exact fp32 path, since fp16 additive masks can overflow to -inf.
        amp = input_ids.is_cuda and (attention_mask is None or bool(attention_mask.bool().all()))
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
            return self._forward_impl(input_ids, attention_mask, past_key_values, labels, use_cache)

    def _forward_impl(self, input_ids, attention_mask, past_key_values, labels, use_cache):
        use_cache = self.config.use_cache if use_cache is None else use_cache
        if self.training:
            use_cache = False  # never keep K/V during training
        B, T = input_ids.shape
        past_len = past_key_values[0][0].shape[2] if past_key_values is not None else 0

        x = self.drop(self.wte(input_ids))
        cos, sin = self._rope_slice(past_len, T, input_ids.device, x.dtype)
        attn_bias, is_causal = self._build_attn_bias(
            attention_mask, past_len, T, input_ids.device, x.dtype
        )

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            pkv = past_key_values[i] if past_key_values is not None else None
            if self.training and self.config.activation_checkpointing:
                x = torch_checkpoint(
                    lambda h, blk=block: blk(h, cos, sin, attn_bias, is_causal, None, False)[0],
                    x,
                    use_reentrant=False,
                )
            else:
                x, present = block(x, cos, sin, attn_bias, is_causal, pkv, use_cache)
                if use_cache:
                    presents.append(present)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )

        # when labels are given only the loss is needed; returning the full
        # (B, T, vocab) logits would just be gathered across GPUs for nothing
        return CausalLMOutputWithPast(
            loss=loss,
            logits=None if labels is not None else logits,
            past_key_values=tuple(presents) if use_cache else None,
        )

    # No prepare_inputs_for_generation / GenerationMixin here on purpose —
    # generation is done by generate_utils.generate_batch(), which calls
    # this model's forward() directly with its own tuple-based KV cache
    # rather than going through model.generate().

In [ ]:
%%writefile /kaggle/working/generate_utils.py
"""
Drop-in replacement for the starter notebook's generate_batch() in
'## 19. Generate 128 Words'. Same signature and return shape, so cell 20
(the prediction loop) and cells 21-22 (prediction.json / submission.zip)
need no changes at all.

Written as a plain per-step loop against the model's own forward() and its
tuple-based KV cache (see legal_gqa_model.py) rather than model.generate(),
to avoid depending on transformers' internal Cache-class machinery.
"""
import torch
import torch.nn.functional as F


def _top_p_filter(logits, top_p):
    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
    probs = F.softmax(sorted_logits, dim=-1)
    cum_probs = torch.cumsum(probs, dim=-1)
    # keep the smallest prefix whose cumulative prob exceeds top_p;
    # always keep at least the top-1 token
    remove = cum_probs > top_p
    remove[..., 1:] = remove[..., :-1].clone()
    remove[..., 0] = False
    sorted_logits[remove] = float("-inf")
    out = torch.full_like(logits, float("-inf"))
    out.scatter_(-1, sorted_idx, sorted_logits)
    return out


def _build_word_token_table(tokenizer):
    """token_id -> lowercased word it produces, only for tokens that start a
    new word (GPT-2 byte-level BPE marks these with a leading space)."""
    table = {}
    for tid in range(len(tokenizer)):
        text = tokenizer.decode([tid])
        if text[:1] in (" ", "\n"):
            word = text.strip().lower()
            if word:
                table.setdefault(word, []).append(tid)
    return {w: torch.tensor(ids) for w, ids in table.items()}


def _verbatim_overlap(prompt_words, continuation_words, min_run=6):
    """True if a >= min_run-word run from the continuation appears verbatim in
    the prompt (case-insensitive). Used only for a post-hoc visibility check --
    the actual prevention happens via trigram seeding below."""
    prompt_joined = " ".join(w.lower() for w in prompt_words)
    for i in range(len(continuation_words) - min_run + 1):
        window = " ".join(w.lower() for w in continuation_words[i:i + min_run])
        if window and window in prompt_joined:
            return window
    return None


@torch.inference_mode()
def generate_batch(
    prompts=None,
    model=None,
    tokenizer=None,
    device=None,
    target_words=128,
    max_new_tokens=384,
    temperature=0.7,
    top_p=0.95,
    tail_window=8,  # tokens looked at when checking for a repeated word
    token_ids=None,  # optional: list of lists of ints (one per prompt) -- skips re-tokenizing
):
    model.eval()
    max_length = model.config.n_positions - max_new_tokens
    if token_ids is not None:
        trunc = [ids[:max_length] for ids in token_ids]  # same right-truncation as the tokenizer path below
        pad_id = tokenizer.pad_token_id
        B = len(trunc)
        maxlen = max(len(ids) for ids in trunc)
        input_ids = torch.full((B, maxlen), pad_id, dtype=torch.long)
        attention_mask = torch.zeros((B, maxlen), dtype=torch.long)
        for i, ids in enumerate(trunc):
            input_ids[i, maxlen - len(ids):] = torch.tensor(ids, dtype=torch.long)  # left-pad, matches tokenizer.padding_side="left"
            attention_mask[i, maxlen - len(ids):] = 1
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
    else:
        inputs = tokenizer(
            prompts, return_tensors="pt", padding=True, truncation=True,
            max_length=max_length,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        input_ids = inputs["input_ids"]
        attention_mask = inputs["attention_mask"]
    input_lengths = attention_mask.sum(dim=1)
    B = input_ids.size(0)

    word_table = getattr(generate_batch, "_word_table_cache", None)
    if word_table is None:
        word_table = _build_word_token_table(tokenizer)
        generate_batch._word_table_cache = word_table  # cache across calls

    # prefill: run the full prompt once to seed the KV cache
    out = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=True)
    past_key_values = out.past_key_values
    next_logits = out.logits[:, -1, :]

    # Real per-batch generation budget, not the fixed max_new_tokens guess -- a fixed
    # cap under-provisions short prompts (which have far more safe room than that) and
    # can still be too little for hard rows where repetition-blocking wastes steps.
    # Bounded by the model's actual trained context length, so this never pushes
    # generation into positions the model was never trained on.
    available_steps = max(1, model.config.n_positions - input_ids.shape[1])

    generated = [[] for _ in range(B)]
    trigram_seen = [set() for _ in range(B)]
    # Seed with the PROMPT's own trigrams too, not just trigrams generated so far --
    # this makes the SAME "block recurrence of any seen trigram" guard also stop the
    # model from reproducing 3+ consecutive tokens verbatim from the prompt (the task's
    # "must not copy the prompt" requirement), reusing an already-tested mechanism
    # rather than adding a new one.
    prompt_ids_per_row = []
    for row in range(B):
        real_ids = input_ids[row][attention_mask[row].bool()].tolist()
        prompt_ids_per_row.append(real_ids)
        for i in range(len(real_ids) - 2):
            trigram_seen[row].add(tuple(real_ids[i:i + 3]))
    finished_word_count = [0] * B
    cur_ids = input_ids
    cur_mask = attention_mask

    for _ in range(available_steps):
        step_logits = next_logits / max(temperature, 1e-5)

        for row in range(B):
            if finished_word_count[row] >= target_words:
                continue

            # --- guard 0: block natural EOS until the target is reached ---
            # Without this, a model trained on EOS-separated documents (exactly
            # what every model in this project is trained on) will sometimes
            # naturally choose EOS mid-continuation when it "feels" a document
            # should end. That token still consumes one of the available_steps
            # loop iterations and gets appended to generated[row], but
            # tokenizer.decode(..., skip_special_tokens=True) at the very end
            # silently strips it -- so it contributes zero words while still
            # spending real budget. Confirmed as the actual mechanism behind a
            # real shortfall (min=74/128 on a production run), not the
            # not-enough-context-room case the WARNING below is actually for.
            if tokenizer.eos_token_id is not None:
                step_logits[row, tokenizer.eos_token_id] = float("-inf")

            # --- guard 1: block a 3rd consecutive repeat of the same word ---
            tail_ids = torch.tensor(generated[row][-tail_window:])
            if tail_ids.numel() > 0:
                tail_text = tokenizer.decode(tail_ids, skip_special_tokens=True)
                words = tail_text.split()
                if len(words) >= 2 and words[-1].lower() == words[-2].lower():
                    block_ids = word_table.get(words[-1].lower())
                    if block_ids is not None:
                        step_logits[row, block_ids.to(device)] = float("-inf")

        # --- guard 2: block recurrence of any trigram already generated ---
        # checked only against each row's current top-50 candidates (the
        # tokens sampling would realistically pick anyway) rather than the
        # full vocab, since this runs every generation step.
        for row in range(B):
            if finished_word_count[row] >= target_words or len(generated[row]) < 2:
                continue
            prefix = (generated[row][-2], generated[row][-1])
            seen = trigram_seen[row]
            # only tokens that would recreate a seen trigram get blocked;
            # we look this up lazily per-row since the seen set is small
            for tid in torch.topk(step_logits[row], k=min(50, step_logits.size(-1))).indices.tolist():
                if (prefix[0], prefix[1], tid) in seen:
                    step_logits[row, tid] = float("-inf")

        filtered = _top_p_filter(step_logits, top_p)
        probs = F.softmax(filtered, dim=-1)
        next_tokens = torch.multinomial(probs, num_samples=1).squeeze(-1)

        for row in range(B):
            if finished_word_count[row] >= target_words:
                next_tokens[row] = tokenizer.eos_token_id
                continue
            tok = next_tokens[row].item()
            generated[row].append(tok)
            if len(generated[row]) >= 3:
                trigram_seen[row].add(tuple(generated[row][-3:]))
            text_so_far = tokenizer.decode(generated[row], skip_special_tokens=True)
            finished_word_count[row] = len(text_so_far.split())

        if all(c >= target_words for c in finished_word_count):
            break

        next_col = next_tokens.unsqueeze(1)
        cur_mask = torch.cat([cur_mask, torch.ones(B, 1, dtype=cur_mask.dtype, device=device)], dim=1)
        out = model(input_ids=next_col, attention_mask=cur_mask, past_key_values=past_key_values, use_cache=True)
        past_key_values = out.past_key_values
        next_logits = out.logits[:, -1, :]

    predictions = []
    for row in range(B):
        text = tokenizer.decode(generated[row], skip_special_tokens=True)
        words = text.split()[:target_words]
        if len(words) < target_words:
            # only happens if this row's prompt was long enough that even the FULL
            # available context budget (n_positions - prompt_length) wasn't enough to
            # reach target_words -- a real limit for the longest prompts, not a bug;
            # printed so it's visible rather than silently submitting a short prediction
            print(f"WARNING: row {row} only reached {len(words)}/{target_words} words "
                  f"(prompt used {input_ids.shape[1]} of {model.config.n_positions} context tokens)")

        prompt_text = tokenizer.decode(prompt_ids_per_row[row], skip_special_tokens=True)
        overlap = _verbatim_overlap(prompt_text.split(), words)
        if overlap is not None:
            # trigram seeding above should make this rare -- printed so it's visible
            # rather than silently submitting a continuation that copies the prompt
            print(f"WARNING: row {row} continuation contains a run copied from the "
                  f"prompt: \"{overlap}\"")

        predictions.append(" ".join(words))
    return predictions

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working")
from legal_gqa_model import LegalGQAConfig, LegalGQAForCausalLM
from generate_utils import generate_batch

## 7. Load The Trained Model

In [ ]:
ckpt_dirs = find_dirs("model.safetensors")
final_dirs = [d for d in ckpt_dirs if d.rstrip("/").endswith("legal-gqa-final") and "checkpoint-" not in d]
if not final_dirs:
    print("No 'legal-gqa-final' folder found. Candidates seen:")
    for d in ckpt_dirs:
        print(" ", d)
    raise FileNotFoundError("Attach model 1's training notebook output (Add Input).")
MODEL_DIR = final_dirs[0]
print("Loading model from:", MODEL_DIR)

model = LegalGQAForCausalLM.from_pretrained(MODEL_DIR).to(device).eval()
print(f"Model size: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")

## 4b. Load Pre-Tokenized Test Set (if attached)

In [ ]:
tok_cache_dirs = find_dirs("test_tokenized.json")
test_token_ids = None
if tok_cache_dirs:
    tok_cache_path = os.path.join(tok_cache_dirs[0], "test_tokenized.json")
    with open(tok_cache_path) as f:
        test_token_ids = json.load(f)  # {example_id: [token ids]}
    print("Using pre-tokenized cache:", tok_cache_path, f"({len(test_token_ids)} examples)")
else:
    print("No pre-tokenized cache attached -- will tokenize prompts inline during generation.")

## 8. Generate Predictions for the Test Set

Uses `generate_batch` as-is (top-p + temperature sampling, the two repetition guards: blocks a 3rd consecutive repeat of the same word, and blocks trigram recurrence). Uses the pre-tokenized cache above if attached (verified to produce byte-identical output to tokenizing inline), otherwise tokenizes from raw text. No verbatim-overlap check against the prompt is implemented here -- worth knowing as a gap, not something silently handled.

In [ ]:
predictions = {}
items = list(test_data.items())
total = len(items)
t0 = time.time()

for start in range(0, total, GENERATION_BATCH_SIZE):
    batch = items[start:start + GENERATION_BATCH_SIZE]
    ids = [str(eid) for eid, _ in batch]

    if test_token_ids is not None:
        gen_kwargs = {"token_ids": [test_token_ids[eid] for eid in ids]}
    else:
        gen_kwargs = {"prompts": [rec["text"] for _, rec in batch]}

    outs = generate_batch(
        model=model, tokenizer=tokenizer, device=device,
        target_words=TARGET_GENERATION_WORDS, max_new_tokens=MAX_GENERATION_TOKENS,
        temperature=TEMPERATURE, top_p=TOP_P, **gen_kwargs,
    )
    for eid, text in zip(ids, outs):
        predictions[eid] = {"text": text}

    done = start + len(batch)
    elapsed = time.time() - t0
    rate = done / elapsed

    eta = (total - done) / rate if rate > 0 else float("inf")
    print(f"{done}/{total} ({100*done/total:.1f}%) | {elapsed/60:.1f} min elapsed | ETA {eta/60:.1f} min")

## 9. Save prediction.json and Create Submission ZIP

In [ ]:
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)
print("Saved:", PREDICTION_PATH)

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")
print("Codabench submission created:", SUBMISSION_ZIP)

# sanity check: word counts should all be <= 128, and none should be empty
lens = [len(v["text"].split()) for v in predictions.values()]
print(f"word count: min={min(lens)} max={max(lens)} mean={sum(lens)/len(lens):.1f}")
print("any empty predictions:", sum(1 for l in lens if l == 0))